In [3]:
# Initialisation
#  echo: false
# Recharge automatiquement les modules modifiés (évite de redémarrer le kernel)
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as sci
from scipy.signal import hilbert
from scipy.stats import kurtosis
from scipy.fftpack import fft, fftfreq
from IPython.display import Markdown

# Cherche le dossier racine du projet (BDS_Fault_Detection) pour faire les imports depuis Sources/
dossier = Path().resolve()
for parent in (dossier, *dossier.parents):
    if "BDS_Fault_Detection" in parent.name:
        RACINE = parent
sys.path.insert(0, str(RACINE))

# Importation des fonctions du projet (dossier Sources/). Attention de l'ordre d'importation  !
from Sources import data, config, signaux, indicateurs, features, models


In [4]:
# Sélection du scope de données à traiter (sources, défauts, diamètres, positions, charges, canaux)
selection = data.selectionner(
    sources=("12kDE", "12kFE", "Normal"),
    defauts=("IR", "B", "OR"),
    diametres=(7, 14, 21), positions_or=("6",),
    charges=(0, 1, 2, 3),
    canaux={"12kDE": ("DE",), 
            "12kFE": ("FE",), 
            "Normal": ("DE","FE")},
)

In [5]:
# Importation des données sélectionnées et ajout des indicateurs temporels et spectraux (ajout de ceux de Mourad)
table = features.construire_table_SpecMourad(selection, data.charger)

In [75]:
# Preparation pour le modeling + modeling
#| label: preparation-modeling
#| code-summary: "Split par fichier + standardisation"
#| tbl-cap: "Répartition des fichiers entre apprentissage et test."
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lazypredict import LazyClassifier

resultats = []
for contexte in ["DE→DE", "DE→FE"]:
    for target in ["defaut", "classe"]:
        for indics in ["Spectre Robin", "Spectre Mourad"]:
            if indics == "Spectre Robin":
                COLONNES_INDICATEURS = indicateurs.COLONNES_TEMPORELS + indicateurs.COLONNES_SPECTRAUX
            elif indics == "Spectre Mourad":
                COLONNES_INDICATEURS = indicateurs.COLONNES_TEMPORELS + indicateurs.COLONNES_Mourad

            if contexte == "DE→DE":
                X_train, X_test, y_train, y_test, g_train, g_test = features.preparer_modeling(table, COLONNES_INDICATEURS, target=target)
            elif contexte == "DE→FE":
                X_train, X_test, y_train, y_test, g_train, g_test = features.preparer_modeling_DE_FE(table, COLONNES_INDICATEURS, canal_train="DE", canal_test="FE", target=target)
            
            X_train, y_train, X_test, y_test = features.equilibrer(X_train, y_train, X_test, y_test, type="train")

            clf = LazyClassifier(verbose=0,ignore_warnings=True, custom_metric=None)
            models,predictions = clf.fit(X_train, X_test, y_train, y_test)
            models

            models["scoreTOT"]   = np.sqrt(models["Balanced Accuracy"] * models["F1 Score"])
            models["scoreTimed"] = models["scoreTOT"]**2 / np.log1p(models["Time Taken"] * 10000)

            # --- enregistrement du meilleur modèle de cette config ---
            best = models["scoreTimed"].idxmax()        # idxmax ignore les NaN
            resultats.append({
                "Train→Test": contexte, "target": target, "features": indics,
                "modele": best,
                **models.loc[best, ["Accuracy", "Balanced Accuracy", "ROC AUC",
                                    "F1 Score", "Time Taken"]].to_dict(),
            })

df_resultats = pd.DataFrame(resultats)
df_resultats

,Train→Test,target,features,modele,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
0,DE→DE,defaut,Spectre Robin,DecisionTreeClassifier,0.952697,0.945112,0.970180,0.952836,0.108437
1,DE→DE,defaut,Spectre Mourad,DecisionTreeClassifier,0.955187,0.952948,0.972399,0.955471,0.092749
2,DE→DE,classe,Spectre Robin,DecisionTreeClassifier,0.846346,0.763000,0.916508,0.829232,0.151295
3,DE→DE,classe,Spectre Mourad,DecisionTreeClassifier,0.813953,0.723000,0.900081,0.803586,0.106731
4,DE→FE,defaut,Spectre Robin,GaussianNB,0.829102,0.844105,0.965507,0.828460,0.084423
5,DE→FE,defaut,Spectre Mourad,NearestCentroid,0.756573,0.780254,0.953319,0.745095,0.077212
6,DE→FE,classe,Spectre Robin,PassiveAggressiveClassifier,0.523572,0.371343,NaN,0.529724,0.082929
7,DE→FE,classe,Spectre Mourad,NearestCentroid,0.449229,0.412224,0.893383,0.472826,0.158330
